# v3.6 SSFP NMR Template

This template runs a no-gradient SSFP-style NMR sequence selected and visualized through the GUI. TR is controlled inside the pulse sequence by the `SSFP_TR_Delay` section and hardware repeat loop, not by Python sleep.

In [ ]:
import importlib
from IPython.display import display

import src_v3_6.app as gui
gui = importlib.reload(gui)

display(gui.GUI_total)

## Select And Inspect The Virgin Sequence

In the GUI, open **Sequence -> Sequence Files**, choose **CSE Sequences**, load `SSFP_NMR_Template.json`, then inspect the timeline/pulse view. The notebook reads the currently loaded GUI sequence as a copy for sanity checks.

In [ ]:
virgin_seq = gui.get_current_sequence()
sections = virgin_seq['SectionConfig']

TR_US = (
    sections['SSFP_RF']['delay']
    + sections['SSFP_RX']['delay']
    + sections['SSFP_TR_Delay']['delay']
)
N_REPETITIONS = virgin_seq['ExpConfig']['cycle_repetition_number']

TR_US, N_REPETITIONS, virgin_seq['ExpConfig']

## Run One Hardware-Controlled SSFP Train

`python_tr_enabled=False` tells the v3.6 runner not to insert Python sleeps. The TR is the sequence timing: `SSFP_RF + SSFP_RX + SSFP_TR_Delay`.

In [ ]:
# Initialise FPGA from the GUI before this cell.
config = gui.make_experiment_config(
    num_averages=1,
    loop_count=1,
    tr_ms=1000,
    python_tr_enabled=False,
)

def on_scan(result):
    print(
        f'SSFP train done: {N_REPETITIONS} sequence-controlled TR periods, '
        f'{result.context.tracing.nr_samples} samples/RX'
    )

avg_ch0, avg_ch1 = gui.run_api_sequence(
    config=config,
    scan_callback=on_scan,
)
avg_ch0.shape, avg_ch1.shape